# LeetCode #87: Scramble String

https://leetcode.com/problems/scramble-string/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n!)$ | $O(n^2)$ |
| **Optimal: Memoized Recursion ★** | $O(n^4)$ | $O(n^3)$ |

---

## Understanding the Methods

### Brute Force
Try every possible way to split and swap sub-strings recursively. Without memoization, the same (s1, s2) pairs are recomputed exponentially — factorial in the string length.

### Optimal: Memoized Recursion ★
A string is a scramble of another if, for some split point $k$, either: (a) `s1[0..k-1]` scrambles to `s2[0..k-1]` and `s1[k..]` scrambles to `s2[k..]`, or (b) `s1[0..k-1]` scrambles to `s2[n-k..]` and `s1[k..]` scrambles to `s2[0..n-k-1]` (swapped halves). A character-frequency early-exit prunes many branches. Results are cached in a 3D memo table keyed by `(i1, i2, length)`, giving $O(n^4)$ time and $O(n^3)$ space.

**Constraints:**
* $1 \leq s1.\text{length} = s2.\text{length} \leq 30$
* `s1` and `s2` consist of lowercase English letters


## Solutions

### C#

In [ ]:
public class Solution {
    private Dictionary<(int, int, int), bool> memo = new();

    public bool IsScramble(string s1, string s2) {
        return Solve(s1, s2, 0, 0, s1.Length);
    }

    private bool Solve(string s1, string s2, int i1, int i2, int len) {
        if (len == 1) return s1[i1] == s2[i2];
        if (memo.TryGetValue((i1, i2, len), out bool cached)) return cached;

        // Prune: frequency mismatch means no scramble is possible
        int[] freq = new int[26];
        for (int i = 0; i < len; i++) {
            freq[s1[i1 + i] - 'a']++;
            freq[s2[i2 + i] - 'a']--;
        }
        foreach (int f in freq) if (f != 0) return memo[(i1, i2, len)] = false;

        for (int k = 1; k < len; k++) {
            // Case 1: no swap — left-left and right-right
            if (Solve(s1, s2, i1, i2, k) && Solve(s1, s2, i1 + k, i2 + k, len - k))
                return memo[(i1, i2, len)] = true;
            // Case 2: swap — left-right and right-left
            if (Solve(s1, s2, i1, i2 + len - k, k) && Solve(s1, s2, i1 + k, i2, len - k))
                return memo[(i1, i2, len)] = true;
        }

        return memo[(i1, i2, len)] = false;
    }
}

### Python

In [ ]:
from functools import lru_cache

class Solution:
    def isScramble(self, s1: str, s2: str) -> bool:
        @lru_cache(maxsize=None)
        def solve(i1: int, i2: int, length: int) -> bool:
            if length == 1:
                return s1[i1] == s2[i2]

            # Prune: frequency mismatch means no scramble is possible
            if sorted(s1[i1:i1+length]) != sorted(s2[i2:i2+length]):
                return False

            for k in range(1, length):
                # Case 1: no swap — left-left and right-right
                if solve(i1, i2, k) and solve(i1+k, i2+k, length-k):
                    return True
                # Case 2: swap — left-right and right-left
                if solve(i1, i2+length-k, k) and solve(i1+k, i2, length-k):
                    return True
            return False

        return solve(0, 0, len(s1))


### Go

In [ ]:
func isScramble(s1 string, s2 string) bool {
    n := len(s1)
    // memo[i1][i2][len] stores 0=unknown, 1=true, 2=false
    memo := make([][][]int8, n)
    for i := range memo {
        memo[i] = make([][]int8, n)
        for j := range memo[i] {
            memo[i][j] = make([]int8, n+1)
        }
    }

    var solve func(i1, i2, length int) bool
    solve = func(i1, i2, length int) bool {
        if length == 1 {
            return s1[i1] == s2[i2]
        }
        if memo[i1][i2][length] != 0 {
            return memo[i1][i2][length] == 1
        }

        // Prune: frequency mismatch means no scramble is possible
        var freq [26]int
        for i := 0; i < length; i++ {
            freq[s1[i1+i]-'a']++
            freq[s2[i2+i]-'a']--
        }
        for _, f := range freq {
            if f != 0 {
                memo[i1][i2][length] = 2
                return false
            }
        }

        for k := 1; k < length; k++ {
            // Case 1: no swap — left-left and right-right
            if solve(i1, i2, k) && solve(i1+k, i2+k, length-k) {
                memo[i1][i2][length] = 1
                return true
            }
            // Case 2: swap — left-right and right-left
            if solve(i1, i2+length-k, k) && solve(i1+k, i2, length-k) {
                memo[i1][i2][length] = 1
                return true
            }
        }

        memo[i1][i2][length] = 2
        return false
    }

    return solve(0, 0, n)
}

### Rust

In [ ]:
use std::collections::HashMap;

impl Solution {
    pub fn is_scramble(s1: String, s2: String) -> bool {
        let s1 = s1.as_bytes();
        let s2 = s2.as_bytes();
        let mut memo: HashMap<(usize, usize, usize), bool> = HashMap::new();
        Self::solve(s1, s2, 0, 0, s1.len(), &mut memo)
    }

    fn solve(
        s1: &[u8], s2: &[u8],
        i1: usize, i2: usize, length: usize,
        memo: &mut HashMap<(usize, usize, usize), bool>,
    ) -> bool {
        if length == 1 { return s1[i1] == s2[i2]; }
        if let Some(&v) = memo.get(&(i1, i2, length)) { return v; }

        // Prune: frequency mismatch means no scramble is possible
        let mut freq = [0i32; 26];
        for i in 0..length {
            freq[(s1[i1 + i] - b'a') as usize] += 1;
            freq[(s2[i2 + i] - b'a') as usize] -= 1;
        }
        if freq.iter().any(|&f| f != 0) {
            memo.insert((i1, i2, length), false);
            return false;
        }

        for k in 1..length {
            // Case 1: no swap — left-left and right-right
            if Self::solve(s1, s2, i1, i2, k, memo)
                && Self::solve(s1, s2, i1+k, i2+k, length-k, memo) {
                memo.insert((i1, i2, length), true);
                return true;
            }
            // Case 2: swap — left-right and right-left
            if Self::solve(s1, s2, i1, i2+length-k, k, memo)
                && Self::solve(s1, s2, i1+k, i2, length-k, memo) {
                memo.insert((i1, i2, length), true);
                return true;
            }
        }

        memo.insert((i1, i2, length), false);
        false
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `s1 = "great", s2 = "rgeat"`
`"great"` splits into `"gr"` and `"eat"`; swapping gives `"rg"` + `"eat"` = `"rgeat"` — true. The no-swap case at split $k=2$ resolves recursively in a few calls.

### 2. Slightly Complex
**Input:** `s1 = "abcde", s2 = "caebd"`
A valid scramble exists via multiple nested splits and swaps. The frequency check passes (same characters), and the memoized recursion finds the match at split $k=3$.

### 3. Edge Case: Time Factor
**Input:** `s1 = s2 = "aaaa...a"` (30 identical characters)
All frequency checks pass trivially; the memo table fills up to $30 \times 30 \times 30 = 27{,}000$ entries — the worst-case state space — but each entry is computed once.

### 4. Edge Case: Space Factor
**Input:** `s1 = "a", s2 = "a"` (length 1)
The base case returns immediately without populating the memo. Space is $O(1)$ effectively — the $O(n^3)$ memo is not populated for trivial inputs.

### 5. Almost-Impossible but Plausible
**Input:** `s1 = "ab", s2 = "ba"`
At $k=1$: case 2 checks `s1[0]` vs `s2[1]` (`'a'` vs `'a'`) and `s1[1]` vs `s2[0]` (`'b'` vs `'b'`) — both match, so result is `true`. This is the smallest non-trivial swap case.
